# Detect items impacted by broken dependencies


Broken dependency is an item that is contained by another item, but isn't valid or accessible. There can be a number of reasons for why these improper dependencies may exist- an item being used by an application mistakenly got deleted, there was some sort of corruption with data, the json structure of an application might have been copied from one in a different org, etc. These sorts of items can lead to rendering or logic issues in applications such as Web Maps, StoryMaps, Experiences, Surveys, and other ArcGIS Apps.

Below, you will build a dependency graph for your items and then detect items impacted by or dependent on broken items, if any. 

> Note: This Template can be used as a diagnostic tool to determine validity of your items, and can be scheduled to run periodically.

## 1. Import libraries and connect to GIS


In [ ]:
from datetime import datetime

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from arcgis.apps.itemgraph import create_dependency_graph
from arcgis.gis import GIS

In [ ]:
gis = GIS("home")

## 2. Retrieve items from content


In [ ]:
my_items = gis.content.search(query=f"owner:{gis.users.me.username}", max_items=-1)
print(f"Total number of items: {len(my_items)}")

## 3. Create dependency graph


In [ ]:
graph = create_dependency_graph(gis, my_items)
all_nodes = graph.all_items()

## 4. Find broken dependencies


Nodes (items) without any attached `item` indicate that they are either inaccessible within this organization or do not exist.


In [ ]:
# Detecting broken nodes from among my items
nodes_without_items = [node for node in all_nodes if not node.item]

We extract all the deep dependencies for these nodes, i.e. we find the items that are impacted by these nodes.

In [ ]:
nodes_weighted = [(node, len(node.required_by()), node.required_by()) for node in nodes_without_items]

We capture details for these deep dependencies in a Pandas DataFrame.

In [ ]:
# Define the dataframe to capture results
df_dependency = pd.DataFrame(
    columns=[
        'broken_item_id',
        'total_count',
        'dependent_item_counter', 
        'dependent_item_id',
        'dependent_item_title', 
        'dependent_item_type', 
        'dependent_item_owner'
    ]
)

# Sort the dependencies in descending order of number of connections
nodes_sorted = sorted(nodes_weighted, key=lambda item: item[1], reverse = True)

# Add details for each node with impacted dependencies to the dataframe
for node, degree, dependencies in nodes_sorted:
    if node.id:
        broken_item_id = node.id
        total_count = degree
        counter = 0
        for dependency in dependencies:
            if dependency.item.id:
                item_id = dependency.item.id
            else:
                item_id = None
            if dependency.item.title:
                title = dependency.item.title
            else:
                title = None
            if dependency.item.type:
                item_type = dependency.item.type
            else:
                item_type = None
            if dependency.item.owner:
                owner = dependency.item.owner
            else:
                owner = None
            counter = counter + 1
            # Add details for each node as a new row
            new_row = {'broken_item_id': broken_item_id, 'total_count': degree, 'dependent_item_counter': counter, 'dependent_item_id': item_id, 
                       'dependent_item_title': title, 'dependent_item_type': item_type, 'dependent_item_owner': owner}
            df_dependency = pd.concat([df_dependency, pd.DataFrame([new_row])], ignore_index=True)

In [ ]:
print(f"My content has {len(nodes_without_items)} broken items with a total of {df_dependency.shape[0]} impacted dependencies.")

In [ ]:
print("Preview (top 10):")
df_dependency.head(10)

## 5. Save report to CSV


In [ ]:
timestamp = datetime.now().strftime("%Y-%m-%d")
file_path = rf"/arcgis/home/broken_dependencies_itemgraph_{timestamp}.csv"
df_dependency.to_csv(file_path)